In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

TUFTS_ROOT = Path("/Users/chennupatigunadeep/Downloads/Capstone/DataSets")
MASK_DIR = TUFTS_ROOT / "Segmentation" / "teeth_mask"

print("=" * 60)
print("TEETH_MASK STRUCTURE INSPECTION")
print("=" * 60)

# Basic structure
items = sorted(MASK_DIR.iterdir())
print(f"Total items: {len(items)}")
print(f"First 10 items: {[item.name for item in items[:10]]}")
print(f"First item is_dir: {items[0].is_dir()}")
print(f"First item is_file: {items[0].is_file()}")

# Detect structure type
if items[0].is_dir():
    print("\nSTRUCTURE: Subfolders (one folder per image)")
    sample_folder = items[0]
    inner = list(sample_folder.iterdir())
    print(f"First folder '{sample_folder.name}' contains {len(inner)} files")
    print(f"First 5 inner files: {[f.name for f in inner[:5]]}")
    first_mask_file = inner[0]
else:
    print("\nSTRUCTURE: Flat (one file per image)")
    extensions = set(f.suffix.lower() for f in items[:100])
    print(f"File extensions: {extensions}")
    first_mask_file = items[0]

# Load and inspect first mask
print(f"\n--- First mask: {first_mask_file.name} ---")
mask = Image.open(first_mask_file)
print(f"Dimensions: {mask.width} × {mask.height}")
print(f"Mode: {mask.mode}")

arr = np.array(mask)
print(f"Array shape: {arr.shape}")
print(f"Array dtype: {arr.dtype}")
print(f"Min/max: {arr.min()} / {arr.max()}")

# Unique values are critical — determines mask type
unique = np.unique(arr)
print(f"Unique values count: {len(unique)}")
print(f"Unique values: {unique[:50]}{'...' if len(unique) > 50 else ''}")

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import pandas as pd
import json

TUFTS_ROOT = Path("/Users/chennupatigunadeep/Downloads/Capstone/DataSets")
MASK_DIR = TUFTS_ROOT / "Segmentation" / "teeth_mask"
ROTATED_IMG_DIR = TUFTS_ROOT / "Radiographs_rotated"
OUTPUT_DIR = Path("outputs/eda_tufts")

def extract_bboxes_from_integer_mask(mask_array):
    """
    For a mask where each tooth has a unique integer label (1, 2, 3, ...),
    extract the tight bounding box for each tooth.
    
    Returns: list of dicts, one per tooth
    """
    unique_labels = np.unique(mask_array)
    unique_labels = unique_labels[unique_labels > 0]  # skip background (0)
    
    bboxes = []
    for label in unique_labels:
        # Find all pixels belonging to this tooth
        ys, xs = np.where(mask_array == label)
        
        if len(xs) == 0 or len(ys) == 0:
            continue
        
        # Tight bounding box = min/max of pixel coordinates
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        
        # Skip tiny/noisy labels (< 10 pixels = likely annotation noise)
        if (x2 - x1) < 2 or (y2 - y1) < 2 or len(xs) < 10:
            continue
        
        bboxes.append({
            "label": int(label),
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_w": x2 - x1,
            "bbox_h": y2 - y1,
            "pixel_count": len(xs),
        })
    
    return bboxes


# Process all masks
all_annotations = []
mask_files = sorted([f for f in MASK_DIR.iterdir() if f.is_file()])

print(f"Processing {len(mask_files)} masks...")

for i, mask_path in enumerate(mask_files):
    if i % 100 == 0:
        print(f"  {i}/{len(mask_files)}")
    
    # Load mask
    mask = Image.open(mask_path)
    
    # IMPORTANT: if masks are landscape, rotate them 90° CCW to match rotated images
    if mask.width > mask.height:
        mask = mask.rotate(90, expand=True)
    
    arr = np.array(mask)
    
    # Extract bboxes
    bboxes = extract_bboxes_from_integer_mask(arr)
    
    # Associate with image filename
    # Masks likely named same as image (e.g., 797.jpg or 797.png)
    image_id = mask_path.stem + ".jpg"  # verify naming convention
    
    for bbox in bboxes:
        bbox["image_file"] = image_id
        all_annotations.append(bbox)

df_mask_bboxes = pd.DataFrame(all_annotations)
df_mask_bboxes.to_csv(OUTPUT_DIR / "annotations_from_masks.csv", index=False)

print(f"\nDone. Total mask-derived bboxes: {len(df_mask_bboxes)}")
print(f"Unique images: {df_mask_bboxes['image_file'].nunique()}")
print(f"Avg teeth per image: {len(df_mask_bboxes) / df_mask_bboxes['image_file'].nunique():.1f}")